In [ ]:
import os 
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scipy.stats as stats
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve
import re
import math
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score, recall_score, f1_score
import joblib
import time
from ultralytics import YOLO
import hida
import keijo
import size_module

In [ ]:
# import itertools
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.svm import SVC
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
# import joblib

# # パラメータ候補
# a_list = np.arange(5, 50, 2)
# # a_list = [20]
# # b: 0.1〜0.5 (0.1刻み)
# b_list = np.arange(0.05, 0.50, 0.02)
# # b_list = [0.3]
# c_list = ["top2average","45rotate"]
# # c_list = ["top2average"]

# results = []
# data = "jikuari_maesyori_gakusyu"

# for a, b, c in itertools.product(a_list, b_list, c_list):
#     try:
#         # 特徴量抽出
#         hida_tappleA = hida.Hida_folder_jikuari(base_dir=f"/home/data/{data}",subfolder="A",n=a,T=b,method=c)
#         result_hidaA = hida_tappleA.run_all()
#         hida_tappleB = hida.Hida_folder_jikuari(base_dir=f"/home/data/{data}",subfolder="B",n=a,T=b,method=c)
#         result_hidaB = hida_tappleB.run_all()
#         hida_tappleC = hida.Hida_folder_jikuari(base_dir=f"/home/data/{data}",subfolder="C",n=a,T=b,method=c)
#         result_hidaC = hida_tappleC.run_all()
        
#         dfA = pd.DataFrame(result_hidaA, columns=["filename", "R"])
#         dfB = pd.DataFrame(result_hidaB, columns=["filename", "R"])
#         dfC = pd.DataFrame(result_hidaC, columns=["filename", "R"])
#         dfA["Label"] = "0"
#         dfB["Label"] = "1"
#         dfC["Label"] = "2"
#         result_hida = pd.concat([dfA, dfB, dfC], axis=0, ignore_index=True)

#         size_tappleA = size_module.Size_folder(base_dir=f"/home/data/{data}",subfolder="A")
#         result_sizeA = size_tappleA.count_white_pixels()
#         size_tappleB = size_module.Size_folder(base_dir=f"/home/data/{data}",subfolder="B")
#         result_sizeB = size_tappleB.count_white_pixels()
#         size_tappleC = size_module.Size_folder(base_dir=f"/home/data/{data}",subfolder="C")
#         result_sizeC = size_tappleC.count_white_pixels()
#         dfA = pd.DataFrame(result_sizeA, columns=["filename", "size_count"])                                                    
#         dfB = pd.DataFrame(result_sizeB, columns=["filename", "size_count"])
#         dfC = pd.DataFrame(result_sizeC, columns=["filename", "size_count"])
#         result_size = pd.concat([dfA, dfB, dfC], axis=0, ignore_index=True)

#         keijo_tappleA = keijo.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="A")
#         result_keijoA = keijo_tappleA.run()
#         keijo_tappleB = keijo.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="B")
#         result_keijoB = keijo_tappleB.run()
#         keijo_tappleC = keijo.Keijo_folder(base_dir=f"/home/data/{data}",subfolder="C")
#         result_keijoC = keijo_tappleC.run()
#         dfA = pd.DataFrame(result_keijoA, columns=["filename", "MSE"])
#         dfB = pd.DataFrame(result_keijoB, columns=["filename", "MSE"])
#         dfC = pd.DataFrame(result_keijoC, columns=["filename", "MSE"])
#         result_keijo = pd.concat([dfA, dfB, dfC], axis=0, ignore_index=True)

#         # 統合
#         df = pd.merge(result_keijo, result_size, on="filename")
#         df = pd.merge(df, result_hida, on="filename")

#         # 学習・評価
#         X = df[["MSE", "size_count", "R"]]
#         y = df["Label"]
#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
#         scaler = StandardScaler()
#         X_train = scaler.fit_transform(X_train)
#         X_test = scaler.transform(X_test)
#         model = SVC(kernel='rbf', C=1.0, gamma='scale', decision_function_shape='ovr')
#         model.fit(X_train, y_train)
#         y_pred = model.predict(X_test)

#         f1 = f1_score(y_test, y_pred, average='macro')
#         results.append({"a": a, "b": b, "c": c, "f1_score": f1})

#         print(f"Finished: a={a}, b={b}, c={c}, f1={f1:.3f}")

#     except Exception as e:
#         print(f"Error with a={a}, b={b}, c={c} → {e}")

# # 上位20件をf1スコアでソート
# top_results = sorted(results, key=lambda x: x["f1_score"], reverse=True)[:20]

# # 保存
# pd.DataFrame(top_results).to_csv("top20_hyperparameter_results.csv", index=False)
# print("Top 20 results saved to 'top20_hyperparameter_results.csv'")


In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import itertools
import hida, size_module, keijo  # 必要に応じてimport調整

data = "jikuari_maesyori_gakusyu"

def objective(trial):
    a = trial.suggest_int("a", 1,40 , step=1)
    b = trial.suggest_float("b", 0.2, 0.6, step=0.01)
    c = trial.suggest_categorical("c", ["top2average", "45rotate"])
    
    try:
        # 襞の特徴
        hida_result = []
        for sub in ["A", "B", "C"]:
            hida_obj = hida.Hida_folder_jikuari(base_dir=f"/home/data/{data}", subfolder=sub, n=a, T=b, method=c)
            hida_result.append(pd.DataFrame(hida_obj.run_all(), columns=["filename", "R"]).assign(Label=str(["A", "B", "C"].index(sub))))
        df_hida = pd.concat(hida_result)

        # サイズ
        size_result = []
        for sub in ["A", "B", "C"]:
            size_obj = size_module.Size_folder(base_dir=f"/home/data/{data}", subfolder=sub)
            size_result.append(pd.DataFrame(size_obj.count_white_pixels(), columns=["filename", "size_count"]))
        df_size = pd.concat(size_result)

        # 形状
        keijo_result = []
        for sub in ["A", "B", "C"]:
            keijo_obj = keijo.Keijo_folder(base_dir=f"/home/data/{data}", subfolder=sub)
            keijo_result.append(pd.DataFrame(keijo_obj.run(), columns=["filename", "MSE"]))
        df_keijo = pd.concat(keijo_result)

        # 統合と学習
        df = pd.merge(df_keijo, df_size, on="filename")
        df = pd.merge(df, df_hida, on="filename")
        X = df[["MSE", "size_count", "R"]]
        y = df["Label"]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        model = SVC(kernel='rbf', C=1.0, gamma='scale')
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        return f1_score(y_test, y_pred, average='macro')

    except Exception as e:
        print(f"Trial failed: a={a}, b={b}, c={c} → {e}")
        return 0.0

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# 結果出力
print("Best parameters:", study.best_params)
print("Best f1_score:", study.best_value)
study.trials_dataframe().to_csv("3optuna_hyperparameter_results.csv", index=False)
